<a href="https://colab.research.google.com/github/julianeguo/exodiscovery-benchmark/blob/main/exodiscover_benchmark_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
from google import genai
import os

GEMINI_API_KEY = userdata.get('GEMINI_KEY_2')
client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# List models (optional)
for model in client.models.list():
  print(f"Model name: {model.name}, Version: {model.version}")

Model name: models/gemini-2.5-flash, Version: 001
Model name: models/gemini-2.5-pro, Version: 2.5
Model name: models/gemini-2.0-flash, Version: 2.0
Model name: models/gemini-2.0-flash-001, Version: 2.0
Model name: models/gemini-2.0-flash-lite-001, Version: 2.0
Model name: models/gemini-2.0-flash-lite, Version: 2.0
Model name: models/gemini-2.5-flash-preview-tts, Version: gemini-2.5-flash-exp-tts-2025-05-19
Model name: models/gemini-2.5-pro-preview-tts, Version: gemini-2.5-pro-preview-tts-2025-05-19
Model name: models/gemma-3-1b-it, Version: 001
Model name: models/gemma-3-4b-it, Version: 001
Model name: models/gemma-3-12b-it, Version: 001
Model name: models/gemma-3-27b-it, Version: 001
Model name: models/gemma-3n-e4b-it, Version: 001
Model name: models/gemma-3n-e2b-it, Version: 001
Model name: models/gemini-flash-latest, Version: Gemini Flash Latest
Model name: models/gemini-flash-lite-latest, Version: Gemini Flash-Lite Latest
Model name: models/gemini-pro-latest, Version: Gemini Pro La

In [ ]:
import pandas as pd

df = pd.read_csv("part1_grouped_dataset.csv")
df.head()

,group_id,anon_id,pl_name,pl_rade,pl_bmasse,pl_insol,label_habitable
0,1,G01_P1,GJ 1002 b,1.030000,1.080000,0.6700,1
1,1,G01_P2,TOI-715 b,1.550000,3.020000,0.6700,1
2,1,G01_P3,WASP-31 b,17.363000,151.916000,970.9417,0
3,1,G01_P4,WASP-132 b,10.099292,136.030558,58.5000,0
4,1,G01_P5,LP 890-9 c,1.367000,25.300000,0.9060,1


In [ ]:
SYSTEM_PROMPT = """
You are given one group of anonymized exoplanets in table form.
Your task is to evaluate each planet independently and determine whether it is habitable.

The data includes the below columns:
* planet radius (pl_rade): radius of the exoplanet, in Earth units
* planet mass (pl_bmasse): mass of the exoplanet, in Earth units
* planet flux (pl_insol): stellar flux of the exoplanet, in Earth units

Evaluation criteria for a habitable exoplanet:
(0.5 <= pl_rade <= 1.6 OR 0.1 <= pl_bmasse <= 3)
AND
0.25 <= pl_insol <= 1.49

Use the numeric values exactly as written in the table. Do not round, approximate, or reinterpret any value.
Do NOT use any tools or code. Do not use external knowledge or prior information about exoplanets.
Base your decision strictly on the criteria above.

Return exactly 5 entries, one for each anon_id in the table.
Format each entry exactly as:

anon_id: <ID>
habitable: YES or NO
key_factors: <2-3 input variables>
explanation: <1 concise sentence>

Do not add headers, group labels, extra commentary, or any other text.

After the 5th entry, write exactly:
END
Do not write anything after END.
"""

In [ ]:
def group_to_table(group_df):
    return group_df[["anon_id","pl_rade","pl_bmasse","pl_insol"]].to_string(index=False)

In [ ]:
from google.genai import types

In [ ]:
def build_prompt(table):
    return SYSTEM_PROMPT + "\n\nDATA:\n" + table + "\n\nANSWER:\n"

def run_model(table):
    prompt = build_prompt(table)

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=450,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
      ),
    )

    print("TEXT:")
    print(repr(response.text))

    print("\nFULL RESPONSE:")
    print(response)

    if response.candidates:
        cand = response.candidates[0]
        print("\nfinish_reason:", cand.finish_reason)
        print("safety_ratings:", getattr(cand, "safety_ratings", None))

        text = response.text.strip()

    if "END" in text:
        text = text.split("END", 1)[0].strip() + "\nEND"

    return text

In [ ]:
import time

results = []

groups = list(df.groupby("group_id"))
total_groups = len(groups)

for i, (gid, group) in enumerate(groups, start=1): #!!Note: modified this to bypass rate limit
    print(f"Running group {i}/{total_groups}")

    table = group_to_table(group)
    prompt = build_prompt(table)
    response = run_model(table)

    results.append({
        "group_id": gid,
        "prompt": prompt,
        "response": response
    })

    print(f"Finished group {i}")

    time.sleep(12)   # about 5 requests per minute

Running group 11/12
TEXT:
'anon_id: G11_P1\nhabitable: NO\nkey_factors: pl_rade, pl_bmasse, pl_insol\nexplanation: The planet radius and mass are outside the habitable range, and the stellar flux is too high.\n\nanon_id: G11_P2\nhabitable: YES\nkey_factors: pl_rade, pl_bmasse, pl_insol\nexplanation: The planet radius, mass, and stellar flux all fall within the specified habitable ranges.\n\nanon_id: G11_P3\nhabitable: NO\nkey_factors: pl_insol\nexplanation: The stellar flux is outside the habitable range, even though the radius and mass are within range.\n\nanon_id: G11_P4\nhabitable: NO\nkey_factors: pl_rade, pl_bmasse, pl_insol\nexplanation: The planet radius and mass are outside the habitable range, and the stellar flux is too high.\n\nanon_id: G11_P5\nhabitable: YES\nkey_factors: pl_rade, pl_bmasse, pl_insol\nexplanation: The planet radius, mass, and stellar flux all fall within the specified habitable ranges.\nEND'

FULL RESPONSE:
sdk_http_response=HttpResponse(
  headers=<dict le

KeyboardInterrupt: 

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("gemini_results_1.csv", index=False)

In [ ]:
SYSTEM_PROMPT_PART_2 = """
You are given one anonymized exoplanet in table form.
Your task is to evaluate whether it is habitable.

The data will include the below columns:
* planet radius (pl_rade): radius of the exoplanet, in Earth units
* planet mass (pl_bmasse): mass of the exoplanet, in Earth units
* planet flux (pl_insol): stellar flux of the exoplanet, in Earth units

Here is the evaluation criteria for a habitable exoplanet:
(0.5 <= pl_rade <= 1.6 OR 0.1 <= pl_bmasse <= 3)
AND
0.25 <= pl_insol <= 1.49

Use the numeric values exactly as written in the table. Do not round, approximate, or reinterpret any value.
Do NOT use any tools or code. Do not use external knowledge or prior information about exoplanets not present in the input.
Base your decision strictly on the numerical criteria above.

Return exactly 1 entry, formatted as:

anon_id: <ID>
habitable: YES or NO
key_factors: <2-3 input variables>
explanation: <1 concise sentence>

Do not add headers, group labels, extra commentary, or any other text.

After answering, write exactly:
END
Do not write anything after END.
"""

In [ ]:
import pandas as pd

orig_df = pd.read_csv("borderline_12.csv")
orig_df.head()
mod_df = pd.read_csv("borderline_12_modified.csv")
mod_df.head()

,pair_id,anon_id,pl_name,pl_rade,pl_bmasse,pl_insol,selected_threshold,label_habitable
0,1,B01,TOI-756 c,1.00,12.87205,0.26,pl_insol@0.25,1
1,2,B02,GJ 1002 c,1.10,1.36000,0.24,pl_insol@0.25,0
2,3,B03,GJ 273 b,1.51,2.89000,1.50,pl_insol@1.49,0
3,4,B04,GJ 273 b,0.88,3.00000,1.49,pl_insol@1.49,1
4,5,B05,Kepler-102 b,0.51,0.05000,1.09,pl_rade@0.50,1


In [ ]:
def row_to_table(row):
    return pd.DataFrame([row])[["anon_id", "pl_rade", "pl_bmasse", "pl_insol"]].to_string(index=False)

In [ ]:
def build_prompt_single(row):
    table = row_to_table(row)
    return "DATA:\n" + table + "\n\nANSWER:\n"

In [ ]:
def run_model_single(row):
    prompt = build_prompt_single(row)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT_PART_2,
            temperature=0,
            max_output_tokens=1024,
        ),
    )

    text = response.text.strip()

    marker = "END"
    if marker in text:
        text = text.split(marker, 1)[0].strip() + "\nEND"

    return prompt, text

In [ ]:
results = []

total_pairs = len(orig_df)

for i in range(total_pairs):
    orig_row = orig_df.iloc[i]
    mod_row = mod_df.iloc[i]

    pair_id = orig_row["pair_id"]
    print(f"Running pair {i+1}/{total_pairs} (pair_id={pair_id})")

    # Original first
    orig_prompt, orig_response = run_model_single(orig_row)
    print("Original response:")
    print(orig_response)
    print("-" * 60)

    results.append({
        "pair_id": pair_id,
        "version": "original",
        "anon_id": orig_row["anon_id"],
        "pl_name": orig_row["pl_name"],
        "pl_rade": orig_row["pl_rade"],
        "pl_bmasse": orig_row["pl_bmasse"],
        "pl_insol": orig_row["pl_insol"],
        "selected_threshold": orig_row["selected_threshold"],
        "label_habitable": orig_row["label_habitable"],
        "prompt": orig_prompt,
        "response": orig_response,
    })

    time.sleep(12)

    # Modified second
    mod_prompt, mod_response = run_model_single(mod_row)
    print("Modified response:")
    print(mod_response)
    print("=" * 60)

    results.append({
        "pair_id": pair_id,
        "version": "modified",
        "anon_id": mod_row["anon_id"],
        "pl_name": mod_row["pl_name"],
        "pl_rade": mod_row["pl_rade"],
        "pl_bmasse": mod_row["pl_bmasse"],
        "pl_insol": mod_row["pl_insol"],
        "selected_threshold": mod_row["selected_threshold"],
        "label_habitable": mod_row["label_habitable"],
        "prompt": mod_prompt,
        "response": mod_response,
    })

    print(f"Finished pair_id={pair_id}")

    time.sleep(12)

Running pair 11/12 (pair_id=11)
Original response:
anon_id: B11
habitable: YES
key_factors: pl_bmasse, pl_insol
explanation: The planet meets the mass and stellar flux criteria for habitability.
END
------------------------------------------------------------
Modified response:
anon_id: B11
habitable: NO
key_factors: pl_rade, pl_bmasse, pl_insol
explanation: The planet's radius and mass are outside the specified habitable range, despite its stellar flux being within range.
END
Finished pair_id=11
Running pair 12/12 (pair_id=12)
Original response:
anon_id: B12
habitable: NO
key_factors: pl_rade, pl_bmasse
explanation: The planet's radius is below the minimum threshold and its mass is above the maximum threshold for habitability.
END
------------------------------------------------------------
Modified response:
anon_id: B12
habitable: YES
key_factors: pl_rade, pl_bmasse, pl_insol
explanation: The planet meets the mass and stellar flux criteria for habitability.
END
Finished pair_id=12


In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("gemini_results_part2_extended.csv", index=False)
results_df.head()

,pair_id,version,anon_id,pl_name,pl_rade,pl_bmasse,pl_insol,selected_threshold,label_habitable,prompt,response
0,11,original,B11,Kepler-895 b,0.22,2.990,1.20,pl_bmasse@3.00,1,DATA:\nanon_id pl_rade pl_bmasse pl_insol\n...,anon_id: B11\nhabitable: YES\nkey_factors: pl_...
1,11,modified,B11,Kepler-895 b,0.22,3.010,1.20,pl_bmasse@3.00,0,DATA:\nanon_id pl_rade pl_bmasse pl_insol\n...,anon_id: B11\nhabitable: NO\nkey_factors: pl_r...
2,12,original,B12,Kepler-895 b,0.41,3.012,1.39,pl_bmasse@3.00,0,DATA:\nanon_id pl_rade pl_bmasse pl_insol\n...,anon_id: B12\nhabitable: NO\nkey_factors: pl_r...
3,12,modified,B12,Kepler-895 b,0.41,2.986,1.39,pl_bmasse@3.00,1,DATA:\nanon_id pl_rade pl_bmasse pl_insol\n...,anon_id: B12\nhabitable: YES\nkey_factors: pl_...
